## Objetivo General

Desarrollar y evaluar modelos de Machine Learning

# Preparación del Entorno

In [1]:
%pip install pandas numpy matplotlib seaborn scikit-learn imbalanced-learn category-encoders xgboost pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
import time

# Validación
from sklearn.model_selection import (
    train_test_split,
    cross_validate,
    RepeatedStratifiedKFold
)

# Preprocesamiento
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer

# Balanceo
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# Modelos supervisados
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import (
    BaggingClassifier,
    VotingClassifier
)

# XGBoost
from xgboost import XGBClassifier

# Métricas
from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay,
    f1_score,
    roc_auc_score
)

# Aprendizaje no supervisado
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

print("✓ Todas las librerías cargadas correctamente")

✓ Todas las librerías cargadas correctamente


# Carga y Partición de Datos

Realizamos una partición estratificada del $80\%$ para entrenamiento y $20\%$ para prueba antes de cualquier imputación o escalamiento de datos. Además, eliminamos la variable dependiente **`Seleccionada`**

In [3]:
# Cargar dataset
df = pd.read_feather('LICIT_depurando_sinNulos.feather')

# Variable objetivo
TARGET = 'Seleccionada'

# Variables predictoras
X = df.drop(columns=[TARGET])

# Variable objetivo
y = df[TARGET]

# División entrenamiento / prueba
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Dataset completo: {df.shape}")
print(f"Instancias de entrenamiento: {X_train.shape[0]}")
print(f"Instancias de prueba: {X_test.shape[0]}")

print("\nDistribución de la variable objetivo:")
print(y.value_counts())

print("\nDistribución porcentual:")
print(y.value_counts(normalize=True).round(4))

Dataset completo: (2806950, 49)
Instancias de entrenamiento: 2245560
Instancias de prueba: 561390

Distribución de la variable objetivo:
Seleccionada
0    2450827
1     356123
Name: count, dtype: int64

Distribución porcentual:
Seleccionada
0    0.8731
1    0.1269
Name: proportion, dtype: float64


## Preprocesamiento

In [4]:
# Todas las variables predictoras son numéricas
numeric_features = X_train.columns.tolist()

print(f"Número de variables predictoras: {len(numeric_features)}")
print("\nTipos de datos:")
print(X_train.dtypes.value_counts())

print("\nValores nulos en X_train:")
print(X_train.isnull().sum().sum())

print("\nValores nulos en X_test:")
print(X_test.isnull().sum().sum())

Número de variables predictoras: 48

Tipos de datos:
int64      41
int32       3
float64     2
Float64     2
Name: count, dtype: int64

Valores nulos en X_train:
906344

Valores nulos en X_test:
226204


### Escalamiento
El escalamiento se incorpora dentro del pipeline de los modelos que lo necesitan para evitar fuga de información. Los árboles de decisión no requieren escalamiento.


In [5]:
# No se ajusta ningún scaler fuera de la validación cruzada.
# Cada pipeline define su propio preprocesamiento.


### Nota metodológica sobre variables y balanceo
No se aplica SMOTE como configuración base. El dataset contiene variables one-hot (`sector_*`, `Tipo_*`, `Region_*`) e identificadores numéricos (`CodigoUnidad`, `CodigoProductoONU`); SMOTE interpola valores y puede generar combinaciones/códigos sintéticos sin interpretación real. Con una clase positiva cercana al 21%, primero se evalúan modelos sin datos sintéticos; Regresión Logística y Árbol usan `class_weight='balanced'`. Además, `CodigoUnidad` y `CodigoProductoONU` son códigos, no magnitudes continuas: conviene estudiar su codificación y posible generalización antes de producción.


# Evaluación Modelos

## Validación Cruzada

Para justificar la selección de algoritmo, entrenamos de forma individual los tres modelos supervisados.
1. **Regresión Logística**: Modelo lineal de alta interpretabilidad.
2. **Árbol de Decisión**: Modelo no lineal ortogonal rígido.
3. **Perceptrón Multicapa**: Red neuronal no lineal regularizada de alta capacidad.

Extraeremos todas las métricas operativas (Precisión, Exactitud, Sensibilidad, F1 y AUC) requeridas por el negocio para tomar una decisión informada.

### Regresión Logística

In [10]:
# 1. Validación Cruzada: Regresión Logística

model_lr = LogisticRegression(
    max_iter=1000,
    C=1.0,
    class_weight='balanced',
    random_state=42
)

scoring_metrics = [
    'accuracy',
    'precision',
    'recall',
    'f1',
    'roc_auc',
    'average_precision'
]

cv = RepeatedStratifiedKFold(
    n_splits=3,
    n_repeats=1,
    random_state=42
)

results_summary = {}

pipeline_lr = ImbPipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler()),
    ('classifier', model_lr)
])

scores_lr = cross_validate(
    pipeline_lr,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring_metrics,
    n_jobs=1,
    pre_dispatch=1,
    return_train_score=False,
    error_score="raise"
)

results_summary['Regresión Logística'] = {
    'Exactitud (Accuracy)': (
        scores_lr['test_accuracy'].mean(),
        scores_lr['test_accuracy'].std()
    ),
    'Precisión (Precision)': (
        scores_lr['test_precision'].mean(),
        scores_lr['test_precision'].std()
    ),
    'Sensibilidad (Recall)': (
        scores_lr['test_recall'].mean(),
        scores_lr['test_recall'].std()
    ),
    'F1-Score': (
        scores_lr['test_f1'].mean(),
        scores_lr['test_f1'].std()
    ),
    'AUC-ROC': (
        scores_lr['test_roc_auc'].mean(),
        scores_lr['test_roc_auc'].std()
    )
}

print("Regresión Logística:")

for metric, (mean, std) in results_summary['Regresión Logística'].items():
    print(f"{metric:<22}: {mean:.4f} (std: {std:.4f})")

MemoryError: Unable to allocate 68.5 MiB for an array with shape (1497040, 48) and data type bool

### Árbol de decisión

In [ ]:
# 2. Validación Cruzada: Árbol de Decisión

model_dt = DecisionTreeClassifier(
    max_depth=8,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42
)

pipeline_dt = ImbPipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('classifier', model_dt)
])

scores_dt = cross_validate(
    pipeline_dt,
    X_train,
    y_train,
    scoring=scoring_metrics,
    cv=cv,
    n_jobs=1
)

results_summary['Árbol de Decisión'] = {
    'Exactitud (Accuracy)': (
        scores_dt['test_accuracy'].mean(),
        scores_dt['test_accuracy'].std()
    ),
    'Precisión (Precision)': (
        scores_dt['test_precision'].mean(),
        scores_dt['test_precision'].std()
    ),
    'Sensibilidad (Recall)': (
        scores_dt['test_recall'].mean(),
        scores_dt['test_recall'].std()
    ),
    'F1-Score': (
        scores_dt['test_f1'].mean(),
        scores_dt['test_f1'].std()
    ),
    'AUC-ROC': (
        scores_dt['test_roc_auc'].mean(),
        scores_dt['test_roc_auc'].std()
    )
}

print("Árbol de Decisión:")

for metric, (mean, std) in results_summary['Árbol de Decisión'].items():
    print(f"{metric:<22}: {mean:.4f} (std: {std:.4f})")

MemoryError: Unable to allocate 234. MiB for an array with shape (41, 748520) and data type int64

### Red Neuronal (MLP)

In [ ]:
# 3. Validación Cruzada: Red Neuronal (MLP)

model_mlp = MLPClassifier(
    hidden_layer_sizes=(100,),
    activation='relu',
    alpha=0.0001,
    max_iter=300,
    random_state=42,
    early_stopping=True
)

pipeline_mlp = ImbPipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler()),
    ('classifier', model_mlp)
])

scores_mlp = cross_validate(
    pipeline_mlp,
    X_train,
    y_train,
    scoring=scoring_metrics,
    cv=cv,
    n_jobs=1
)

results_summary['Red Neuronal (MLP)'] = {
    'Exactitud (Accuracy)': (
        scores_mlp['test_accuracy'].mean(),
        scores_mlp['test_accuracy'].std()
    ),
    'Precisión (Precision)': (
        scores_mlp['test_precision'].mean(),
        scores_mlp['test_precision'].std()
    ),
    'Sensibilidad (Recall)': (
        scores_mlp['test_recall'].mean(),
        scores_mlp['test_recall'].std()
    ),
    'F1-Score': (
        scores_mlp['test_f1'].mean(),
        scores_mlp['test_f1'].std()
    ),
    'AUC-ROC': (
        scores_mlp['test_roc_auc'].mean(),
        scores_mlp['test_roc_auc'].std()
    )
}

print("Red Neuronal (MLP):")

for metric, (mean, std) in results_summary['Red Neuronal (MLP)'].items():
    print(f"{metric:<22}: {mean:.4f} (std: {std:.4f})")

boxplot comparacion en validacion cruzada

In [ ]:
# Comparación de F1-Score mediante validación cruzada

f1_results = [
    scores_lr['test_f1'],
    scores_dt['test_f1'],
    scores_mlp['test_f1']
]

names = [
    'Regresión Logística',
    'Árbol de Decisión',
    'Red Neuronal (MLP)'
]

plt.figure(figsize=(10, 6))

plt.boxplot(
    f1_results,
    labels=names,
    showmeans=True
)

plt.title(
    'Comparativa de Algoritmos - F1-Score'
)

plt.ylabel('F1-Score')
plt.xlabel('Algoritmo')
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.show()

Tabla comparativa de métricas por modelo para la clase positiva (`Seleccionada = 1`).


In [ ]:
tabla_metricas = pd.DataFrame({
    modelo: {
        'Exactitud': results_summary[modelo]['Exactitud (Accuracy)'][0],
        'Precisión': results_summary[modelo]['Precisión (Precision)'][0],
        'Sensibilidad': results_summary[modelo]['Sensibilidad (Recall)'][0],
        'F1': results_summary[modelo]['F1-Score'][0],
        'AUC': results_summary[modelo]['AUC-ROC'][0]
    }
    for modelo in results_summary
}).T

tabla_metricas = tabla_metricas.round(4)

display(tabla_metricas)

## Evaluación final en el conjunto de prueba y guardado
El conjunto de prueba se utiliza una sola vez, después de seleccionar el modelo mediante validación cruzada. Esto permite estimar el rendimiento sobre datos no usados para seleccionar el algoritmo.


In [ ]:
# Seleccionar automáticamente el pipeline con mayor F1 medio en validación cruzada
pipelines = {
    'Regresión Logística': pipeline_lr,
    'Árbol de Decisión': pipeline_dt,
    'Red Neuronal (MLP)': pipeline_mlp
}

best_name = max(results_summary, key=lambda m: results_summary[m]['F1-Score'][0])
best_pipeline = pipelines[best_name]

# Ajustar el modelo elegido con TODO el conjunto de entrenamiento
best_pipeline.fit(X_train, y_train)

# Evaluación final sobre test (no utilizado durante la selección)
y_pred = best_pipeline.predict(X_test)
y_prob = best_pipeline.predict_proba(X_test)[:, 1]

print(f'Modelo seleccionado por F1-CV: {best_name}')
print('\nClassification report (TEST):')
print(classification_report(y_test, y_pred, digits=4))
print(f'F1 TEST: {f1_score(y_test, y_pred):.4f}')
print(f'AUC-ROC TEST: {roc_auc_score(y_test, y_prob):.4f}')

ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title(f'Matriz de confusión - {best_name} (TEST)')
plt.show()

# Guardar el pipeline completo ya entrenado
filename = 'modelo_seleccionado.sav'
with open(filename, 'wb') as f:
    pickle.dump(best_pipeline, f)

print(f'Pipeline guardado en: {filename}')
